# 🚀 Tasc Interactive Demo

**Tasc** = Task Automation and Safety Certification

This notebook demonstrates a realistic use case: **Safely debugging a failing test while tracking evidence**.

## What we'll do:
1. Create a checkpoint (safety net)
2. Run terminal commands with safety checks
3. Track intent and reality in ledgers
4. Use the Overlord to check dangerous commands
5. Export an audit trail
6. Rollback if needed

In [ ]:
# Setup - import everything we need
import os
import sys
import tempfile
from datetime import datetime

# Make sure we can import tasc
sys.path.insert(0, os.path.dirname(os.getcwd()))

print("✅ Setup complete!")

: 

## 1️⃣ Load the Action Registry

Tasc has 32 predefined actions with metadata about safety, permissions, and rollback support.

In [ ]:
from tascer.action_registry import get_registry

registry = get_registry()
registry.load()

print(f"📦 Loaded {len(registry.list_all())} actions")
print(f"   - Observations: {len(registry.list_observations())}")
print(f"   - Mutations: {len(registry.list_mutations())}")
print()

# Show some actions
print("Sample actions:")
for action in list(registry.list_all())[:5]:
    meta = registry.get_action(action)
    print(f"  • {action}: {meta.description[:50]}...")

## 2️⃣ Create a Checkpoint (Safety Net)

Before making any changes, we create a checkpoint. This captures the current state so we can rollback if something goes wrong.

In [ ]:
from tascer.checkpoint import CheckpointManager

# Create a temp directory for our demo
demo_dir = tempfile.mkdtemp(prefix="tasc_demo_")

# Create some "project files"
with open(os.path.join(demo_dir, "app.py"), "w") as f:
    f.write("# Main app\nprint('Hello')\n")

with open(os.path.join(demo_dir, "test_app.py"), "w") as f:
    f.write("# Tests\ndef test_hello():\n    assert True\n")

# Create checkpoint manager
checkpoint_mgr = CheckpointManager(
    run_id="notebook_demo",
    root_dir=demo_dir,
    output_dir=demo_dir,
)

# Create the checkpoint
checkpoint = checkpoint_mgr.create("Before debugging session")

print(f"✅ Checkpoint created: {checkpoint.checkpoint_id}")
print(f"   Files tracked: {list(checkpoint.file_snapshot.keys())}")
print(f"   Git commit: {checkpoint.git_state.get('commit', 'N/A')[:8] if checkpoint.git_state else 'N/A'}")

## 3️⃣ Initialize Dual Ledgers

Tasc uses two ledgers:
- **Exe Ledger**: Records *intent* (what we plan to do and why)
- **Moments Ledger**: Records *reality* (what actually happened)

In [ ]:
from tascer.ledgers import LedgerStorage
from tascer.ledgers.exe import ConfidenceScore

storage = LedgerStorage(run_id="notebook_demo", output_dir=demo_dir)

# Record initial context
storage.moments.record_context({
    "cwd": demo_dir,
    "purpose": "Debug failing test",
    "timestamp": datetime.now().isoformat(),
})

# Record our intent
storage.exe.record_narrative(
    "Starting debugging session. Hypothesis: test is failing due to import error."
)

print(f"📚 Ledgers initialized")
print(f"   Moments entries: {len(storage.moments)}")
print(f"   Exe entries: {len(storage.exe)}")

## 4️⃣ Check Command Safety (Overlord)

Before running any command, the Overlord checks if it's safe.

In [ ]:
from tascer.overlord.legality import check_action_legality

# Test various commands
test_commands = [
    "python -m pytest test_app.py",
    "cat app.py",
    "rm -rf /",
    "curl https://api.example.com/data",
    "sudo rm -rf ~/.ssh",
]

print("🛡️ Safety Check Results:")
print("=" * 50)

for cmd in test_commands:
    result = check_action_legality(
        action_id="terminal.run",
        inputs={"command": cmd},
        permissions={"terminal"},
        has_checkpoint=True,
    )
    
    status = "✅ SAFE" if result.is_legal else "🚫 BLOCKED"
    print(f"{status}: {cmd[:40]}")
    if not result.is_legal:
        print(f"         Reason: {result.violations[0][:40]}")

## 5️⃣ Execute Commands with Recording

Now let's run some safe commands and record everything.

In [ ]:
from tascer.primitives import run_and_observe

# Record intent
storage.exe.record_proposal(
    action_id="terminal.run",
    narrative="Running test to see the failure",
    confidence=ConfidenceScore(value=0.8, calibration_note="Standard diagnostic")
)

# Execute command
result = run_and_observe(
    f"cd {demo_dir} && echo 'Running tests...' && ls -la",
    shell=True,
)

# Record reality
storage.moments.record_action_start("terminal.run", {"command": "ls -la"})
storage.moments.record_action_result("terminal.run", {
    "exit_code": result.exit_code,
    "stdout": result.stdout[:200],
    "duration_ms": result.duration_ms,
})

print(f"▶️ Command executed")
print(f"   Exit code: {result.exit_code}")
print(f"   Duration: {result.duration_ms:.1f}ms")
print(f"   Output:")
print(result.stdout)

## 6️⃣ Make Changes (With Tracking)

In [ ]:
# Record intent to modify file
storage.exe.record_proposal(
    action_id="file.write",
    narrative="Fixing the app.py file",
    confidence=ConfidenceScore(value=0.9, calibration_note="Simple fix")
)

# Make the change
new_content = '''# Main app - FIXED!
def hello():
    return "Hello, World!"

if __name__ == "__main__":
    print(hello())
'''

with open(os.path.join(demo_dir, "app.py"), "w") as f:
    f.write(new_content)

# Record the mutation
storage.moments.record_action_result("file.write", {
    "path": "app.py",
    "bytes_written": len(new_content),
    "status": "modified",
})

print("✏️ File modified!")
print(f"   Written {len(new_content)} bytes to app.py")

## 7️⃣ Demonstrate Rollback

Oops! Let's pretend we made a mistake and need to rollback.

In [ ]:
# Show current content
print("📄 Current app.py:")
with open(os.path.join(demo_dir, "app.py")) as f:
    print(f.read())

# Rollback!
print("\n⏪ Rolling back...")
rollback_result = checkpoint_mgr.rollback()

print(f"   Files restored: {len(rollback_result['files_restored'])}")

# Show restored content
print("\n📄 Restored app.py:")
with open(os.path.join(demo_dir, "app.py")) as f:
    print(f.read())

## 8️⃣ Export Audit Trail

Generate a markdown report of everything that happened.

In [ ]:
from tascer.ledgers.exe import StopReason
from tascer.audit import export_to_markdown

# Record completion
storage.exe.record_stop(StopReason.GOAL_ACHIEVED, "Demo complete, rolled back successfully")

# Export
audit_path = export_to_markdown(
    storage=storage,
    output_dir=demo_dir,
    hypothesis="Demonstrate Tasc safety features",
)

print(f"📝 Audit report: {audit_path}")
print("\n" + "=" * 50)
print("Preview:")
print("=" * 50)

with open(audit_path) as f:
    lines = f.readlines()[:25]
    for line in lines:
        print(line.rstrip())

## 9️⃣ Browser Capture (Bonus!)

Tasc can also capture web pages with Playwright.

In [ ]:
try:
    from tascer.primitives.browser import browser_capture, browser_close
    
    state = browser_capture(
        url="https://example.com",
        capture_screenshot=True,
        capture_dom=True,
        screenshot_dir=demo_dir,
    )
    
    browser_close()
    
    print(f"📸 Captured!")
    print(f"   URL: {state.url}")
    print(f"   Title: {state.title}")
    print(f"   Screenshot: {state.screenshot_path}")
    print(f"   DOM size: {len(state.dom_snapshot)} chars")
    
except Exception as e:
    print(f"⚠️ Browser capture skipped: {e}")

## 🔟 Plugin System Preview

In [ ]:
from tascer.plugins import PluginRegistry
from tascer.plugins.mcp_plugin import MCPPlugin
from tascer.plugins.metrics_plugin import MetricsPlugin

reg = PluginRegistry()

mcp = MCPPlugin()
mcp._register_tools()
reg.register(mcp)

metrics = MetricsPlugin()
reg.register(metrics)

print("🔌 Available Plugins:")
for p in reg.list_plugins():
    print(f"   • {p.name} v{p.version}: {p.description}")

print("\n🤖 MCP Tools (for Claude Code):")
for tool in mcp.get_mcp_manifest()["tools"]:
    print(f"   • {tool['name']}")

## ✅ Summary

In this demo, we:

1. **Loaded 32 actions** from the action registry
2. **Created a checkpoint** before making changes
3. **Set up dual ledgers** to track intent and reality
4. **Checked command safety** with the Overlord
5. **Executed commands** with full recording
6. **Made and tracked file changes**
7. **Rolled back** to restore original state
8. **Exported an audit trail** in markdown
9. **Captured a web page** with Playwright
10. **Previewed the plugin system**

### Key Features:
- 🛡️ **Safety**: Dangerous commands are blocked
- 📚 **Traceability**: Every action is recorded
- ⏪ **Rollback**: Mistakes can be undone
- 📝 **Audit**: Markdown reports for evidence
- 🔌 **Extensible**: Plugin system for integrations

In [ ]:
# Cleanup
import shutil
shutil.rmtree(demo_dir, ignore_errors=True)
print("🧹 Cleaned up demo files")